Modelo churn V4: balanceado + auditoría fuerte de fuga + poda iterativa.

Este script hace:

1. Carga el dataset Gold exportado a CSV.
2. Crea la variable objetivo churn.
3. Separa train/test antes de cualquier balanceo.
4. Audita fuga de información usando SOLO train.
5. Detecta fuga en:
   - nombres sospechosos,
   - variables directas,
   - variables numéricas con AUC univariado alto,
   - variables categóricas con tasas de churn casi perfectas,
   - variables numéricas de baja cardinalidad tratadas como categóricas.
6. Hace balanceo de clases SOLO en train.
7. Hace una poda iterativa de variables que siguen generando AUC sospechosamente alto.
8. Entrena varios modelos.
9. Evalúa sobre test original, sin balancear.
10. Genera métricas clásicas, ranking, matrices, predicciones e importancia de variables.

Ejecutar desde la raíz del proyecto:


In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path
from typing import Any

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


warnings.filterwarnings("ignore")

In [ ]:

# ============================================================
# Configuración general
# ============================================================

RUTA_DATASET = Path("../notebooks/churn_prediction_features.csv")

CARPETA_REPORTES = Path("reports_v4_balanceado_auditado")
CARPETA_SALIDAS = Path("outputs_v4_balanceado_auditado")
CARPETA_MODELOS = Path("models_v4_balanceado_auditado")

RANDOM_STATE = 42
TEST_SIZE = 0.20
VALIDATION_SIZE_FOR_PRUNING = 0.25

COLUMNA_TARGET_ORIGEN = "ha_solicitado_retiro"
COLUMNA_LABEL = "churn"

# Balanceo de train:
# opciones: "none", "undersample", "oversample"
BALANCE_METHOD = "undersample"

# Auditoría de fuga
MIN_CANTIDAD_CATEGORIA = 100
UMBRAL_AUC_FUGA_NUMERICA = 0.90
UMBRAL_AUC_FUGA_MODELO_UNIVARIADO = 0.90
UMBRAL_TASA_ALTA_FUGA = 0.98
UMBRAL_TASA_BAJA_FUGA = 0.02
LOW_CARDINALITY_AS_CATEGORICAL_NUNIQUE = 25

# Poda iterativa
ACTIVAR_PODA_ITERATIVA = True
AUC_ALERTA_PODA = 0.90
AUC_OBJETIVO_PODA = 0.85
MAX_ITERACIONES_PODA = 10
MIN_VARIABLES_DESPUES_PODA = 7

TOP_PORCENTAJES = [0.01, 0.05, 0.10, 0.20]


In [ ]:
# ============================================================
# Variables
# ============================================================

COLUMNAS_EXCLUIR_SIEMPRE = [
    # Identificadores
    "cliente_id",
    "Nit",
    "nit",
    "id_cliente",
    "cedula",
    "documento",

    # Target
    COLUMNA_LABEL,
    COLUMNA_TARGET_ORIGEN,

    # Retiro directo
    "retiro_valor_total",
    "retiro_motivo",
    "retiro_tendencia_saldo",

    # Scores o segmentos ya calculados con lógica de churn
    "churn_risk_score",
    "churn_risk_segment",

    # Recencia / engagement derivados
    "recencia_dias_min",
    "recencia_dias_max",
    "recencia_segment",
    "engagement_score",
    "segmento_engagement",

    # Indicadores demasiado cercanos al evento
    "inactivo_transaccional",
    "sin_productos_ahorro",
    "variacion_saldo_negativa",

    # Metadata
    "timestamp_ingestion",
    "load_date",
    "source_file",
]

PATRONES_EXCLUIR_SIEMPRE = [
    "churn",
    "retiro",
    "risk",
    "riesgo",
    "recencia",
    "engagement",
]

# Variables que, conceptualmente, podrían ser previas si fueron calculadas antes del evento.
# El script NO las acepta automáticamente: primero pasan por auditoría de fuga.
COLUMNAS_CANDIDATAS_PREVIAS = [
    # Demográficas
    "sexo",
    "edad",
    "nivel_ingresos",

    # Antigüedad
    "antiguedad_total_meses",
    "antiguedad_anos",
    "permanencia_meses",
    "segmento_antiguedad",

    # Comportamiento transaccional
    "frecuencia_tx_total",
    "frecuencia_tx_mensual_avg",
    "actividad_transaccional",

    # Saldos / valor
    "saldo_total",
    "variacion_saldo_6m",
    "segmento_valor",

    # Productos / diversificación
    "num_productos",
    "nivel_diversificacion_num",
    "segmento_diversificacion",

    # Canales
    "usa_canal_digital",
    "visita_fisica_agencia",
    "es_multicanal",
    "canal_preferido",

    # Otros
    "tiene_castigo",
    "participacion_social",
    "uso_creditos",
]

# Variables muy conservadoras para un modelo de control.
COLUMNAS_ESTRICTAS = [
    "sexo",
    "edad",
    "nivel_ingresos",
    "antiguedad_total_meses",
    "antiguedad_anos",
    "permanencia_meses",
    "segmento_antiguedad",
]

In [ ]:
# ============================================================
# Utilidades base
# ============================================================

def crear_carpetas() -> None:
    CARPETA_REPORTES.mkdir(exist_ok=True)
    CARPETA_SALIDAS.mkdir(exist_ok=True)
    CARPETA_MODELOS.mkdir(exist_ok=True)


def cargar_datos() -> pd.DataFrame:
    if not RUTA_DATASET.exists():
        raise FileNotFoundError(
            f"No se encontró el archivo {RUTA_DATASET}. "
            "Ejecuta este script desde la raíz del proyecto."
        )

    df = pd.read_csv(RUTA_DATASET, low_memory=False)

    print("\nDataset cargado")
    print(f"Filas: {df.shape[0]:,}")
    print(f"Columnas: {df.shape[1]:,}")

    return df



def construir_label(df: pd.DataFrame) -> pd.DataFrame:
    if COLUMNA_TARGET_ORIGEN not in df.columns:
        raise ValueError(f"No existe la columna target origen: {COLUMNA_TARGET_ORIGEN}")

    df = df.copy()

    df[COLUMNA_LABEL] = (
        pd.to_numeric(df[COLUMNA_TARGET_ORIGEN], errors="coerce")
        .fillna(0)
        .astype(int)
    )

    print("\nDistribución del target:")
    print(df[COLUMNA_LABEL].value_counts())

    print("\nDistribución porcentual:")
    print((df[COLUMNA_LABEL].value_counts(normalize=True) * 100).round(2))

    return df


def columna_excluida_por_patron(columna: str) -> bool:
    nombre = columna.lower()
    return any(patron in nombre for patron in PATRONES_EXCLUIR_SIEMPRE)


def crear_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
            max_categories=30,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )

In [ ]:
# ============================================================
# Split y balanceo
# ============================================================

def separar_train_test(
    df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_df, test_df = train_test_split(
        df,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=df[COLUMNA_LABEL],
    )

    print("\nSplit train/test creado")
    print(f"Train: {train_df.shape[0]:,}")
    print(f"Test: {test_df.shape[0]:,}")

    print("\nDistribución target en train:")
    print(train_df[COLUMNA_LABEL].value_counts(normalize=True).round(4))

    print("\nDistribución target en test:")
    print(test_df[COLUMNA_LABEL].value_counts(normalize=True).round(4))

    return train_df.copy(), test_df.copy()


def balancear_train(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    metodo: str = BALANCE_METHOD,
) -> tuple[pd.DataFrame, pd.Series]:
    """
    Balancea SOLO el conjunto de entrenamiento.

    Importante:
    El test NO se balancea, porque debe representar la distribución real.
    """

    if metodo == "none":
        print("\nBalanceo desactivado.")
        return X_train.copy(), y_train.copy()

    df_train = X_train.copy()
    df_train[COLUMNA_LABEL] = y_train.values

    df_0 = df_train[df_train[COLUMNA_LABEL] == 0]
    df_1 = df_train[df_train[COLUMNA_LABEL] == 1]

    if df_0.empty or df_1.empty:
        raise ValueError("No se puede balancear porque una clase está vacía.")

    if metodo == "undersample":
        n = min(len(df_0), len(df_1))

        df_0_sample = df_0.sample(n=n, random_state=RANDOM_STATE)
        df_1_sample = df_1.sample(n=n, random_state=RANDOM_STATE)

        df_bal = pd.concat([df_0_sample, df_1_sample], axis=0)

    elif metodo == "oversample":
        n = max(len(df_0), len(df_1))

        df_0_sample = df_0.sample(
            n=n,
            replace=len(df_0) < n,
            random_state=RANDOM_STATE,
        )

        df_1_sample = df_1.sample(
            n=n,
            replace=len(df_1) < n,
            random_state=RANDOM_STATE,
        )

        df_bal = pd.concat([df_0_sample, df_1_sample], axis=0)

    else:
        raise ValueError("metodo debe ser 'none', 'undersample' u 'oversample'.")

    df_bal = df_bal.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

    y_bal = df_bal[COLUMNA_LABEL].copy()
    X_bal = df_bal.drop(columns=[COLUMNA_LABEL])

    print(f"\nBalanceo aplicado SOLO en train: {metodo}")
    print(y_bal.value_counts())
    print((y_bal.value_counts(normalize=True) * 100).round(2))

    return X_bal, y_bal

In [ ]:
# ============================================================
# Auditoría de fuga
# ============================================================

def calcular_auc_numerico(
    serie: pd.Series,
    y: pd.Series,
) -> float | None:
    datos = pd.DataFrame({"x": serie, "y": y}).dropna()

    if datos.empty or datos["x"].nunique(dropna=False) <= 1:
        return None

    try:
        auc = roc_auc_score(datos["y"], datos["x"])
        return float(max(auc, 1 - auc))
    except ValueError:
        return None


def auditar_como_categorica(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    columna: str,
) -> dict[str, Any]:
    tmp = pd.DataFrame(
        {
            columna: X_train[columna],
            COLUMNA_LABEL: y_train.values,
        }
    )

    tabla = (
        tmp.groupby(columna, dropna=False)[COLUMNA_LABEL]
        .agg(["count", "mean"])
        .reset_index()
    )

    tabla_filtrada = tabla[tabla["count"] >= MIN_CANTIDAD_CATEGORIA].copy()

    if tabla_filtrada.empty:
        return {
            "max_tasa_churn": None,
            "min_tasa_churn": None,
            "categoria_max": None,
            "categoria_min": None,
            "fuga_por_tasa": False,
            "motivo_tasa": "",
        }

    fila_max = tabla_filtrada.loc[tabla_filtrada["mean"].idxmax()]
    fila_min = tabla_filtrada.loc[tabla_filtrada["mean"].idxmin()]

    max_tasa = float(fila_max["mean"])
    min_tasa = float(fila_min["mean"])

    fuga = False
    motivo = ""

    if max_tasa >= UMBRAL_TASA_ALTA_FUGA:
        fuga = True
        motivo = f"categoria_tasa_churn_muy_alta_{max_tasa:.4f}"

    if min_tasa <= UMBRAL_TASA_BAJA_FUGA:
        fuga = True
        if motivo:
            motivo += "|"
        motivo += f"categoria_tasa_churn_muy_baja_{min_tasa:.4f}"

    return {
        "max_tasa_churn": max_tasa,
        "min_tasa_churn": min_tasa,
        "categoria_max": str(fila_max[columna]),
        "categoria_min": str(fila_min[columna]),
        "fuga_por_tasa": fuga,
        "motivo_tasa": motivo,
    }


def crear_preprocesador_univariado(columna: str, es_numerica: bool) -> ColumnTransformer:
    if es_numerica:
        return ColumnTransformer(
            transformers=[
                (
                    "num",
                    Pipeline(
                        steps=[
                            ("imputer", SimpleImputer(strategy="median")),
                            ("scaler", StandardScaler()),
                        ]
                    ),
                    [columna],
                )
            ],
            remainder="drop",
        )

    return ColumnTransformer(
        transformers=[
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", crear_one_hot_encoder()),
                    ]
                ),
                [columna],
            )
        ],
        remainder="drop",
    )


def auc_modelo_univariado(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    columna: str,
) -> float | None:
    """
    Evalúa si una sola variable permite predecir demasiado bien el target.

    Se hace split interno dentro del train.
    No toca el test final.
    """

    X_col = X_train[[columna]].copy()
    y = y_train.copy()

    if X_col[columna].nunique(dropna=False) <= 1:
        return None

    X_a, X_b, y_a, y_b = train_test_split(
        X_col,
        y,
        test_size=0.30,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    es_numerica = pd.api.types.is_numeric_dtype(X_col[columna])

    modelo = Pipeline(
        steps=[
            ("preprocesador", crear_preprocesador_univariado(columna, es_numerica)),
            (
                "modelo",
                LogisticRegression(
                    max_iter=500,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

    try:
        modelo.fit(X_a, y_a)
        proba = modelo.predict_proba(X_b)[:, 1]
        auc = roc_auc_score(y_b, proba)
        return float(max(auc, 1 - auc))
    except Exception:
        return None


def auditar_variables_train(
    train_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Audita fuga usando SOLO train.
    """

    y_train = train_df[COLUMNA_LABEL].copy()

    columnas_a_revisar = [
        c for c in train_df.columns
        if c != COLUMNA_LABEL
    ]

    registros = []

    for columna in columnas_a_revisar:
        serie = train_df[columna]

        excluida_exacta = columna in COLUMNAS_EXCLUIR_SIEMPRE
        excluida_patron = columna_excluida_por_patron(columna)

        registro: dict[str, Any] = {
            "variable": columna,
            "tipo": str(serie.dtype),
            "nulos_pct": float(serie.isna().mean()),
            "n_unicos": int(serie.nunique(dropna=False)),
            "auc_numerico": None,
            "auc_modelo_univariado": None,
            "max_tasa_churn": None,
            "min_tasa_churn": None,
            "categoria_max": None,
            "categoria_min": None,
            "posible_fuga": False,
            "motivos_fuga": "",
            "excluida_exacta": excluida_exacta,
            "excluida_patron": excluida_patron,
        }

        motivos = []

        if excluida_exacta:
            motivos.append("exclusion_exacta")

        if excluida_patron:
            motivos.append("exclusion_por_patron_nombre")

        es_numerica = pd.api.types.is_numeric_dtype(serie)

        if es_numerica:
            auc_num = calcular_auc_numerico(serie, y_train)
            registro["auc_numerico"] = auc_num

            if auc_num is not None and auc_num >= UMBRAL_AUC_FUGA_NUMERICA:
                motivos.append(f"auc_numerico_alto_{auc_num:.4f}")

        # Muy importante:
        # Las numéricas con pocos valores también se auditan como categóricas.
        # Esto captura variables como num_productos, tiene_castigo, usa_canal_digital.
        auditar_tasa = (
            not es_numerica
            or serie.nunique(dropna=False) <= LOW_CARDINALITY_AS_CATEGORICAL_NUNIQUE
        )

        if auditar_tasa:
            resultado_cat = auditar_como_categorica(
                X_train=train_df,
                y_train=y_train,
                columna=columna,
            )

            registro.update(
                {
                    "max_tasa_churn": resultado_cat["max_tasa_churn"],
                    "min_tasa_churn": resultado_cat["min_tasa_churn"],
                    "categoria_max": resultado_cat["categoria_max"],
                    "categoria_min": resultado_cat["categoria_min"],
                }
            )

            if resultado_cat["fuga_por_tasa"]:
                motivos.append(resultado_cat["motivo_tasa"])

        auc_uni = auc_modelo_univariado(
            X_train=train_df,
            y_train=y_train,
            columna=columna,
        )

        registro["auc_modelo_univariado"] = auc_uni

        if auc_uni is not None and auc_uni >= UMBRAL_AUC_FUGA_MODELO_UNIVARIADO:
            motivos.append(f"auc_modelo_univariado_alto_{auc_uni:.4f}")

        if motivos:
            registro["posible_fuga"] = True
            registro["motivos_fuga"] = "|".join(motivos)

        registros.append(registro)

    auditoria = pd.DataFrame(registros)

    auditoria = auditoria.sort_values(
        by=[
            "posible_fuga",
            "auc_modelo_univariado",
            "auc_numerico",
            "max_tasa_churn",
        ],
        ascending=[False, False, False, False],
    )

    ruta = CARPETA_REPORTES / "auditoria_fuga_train.csv"
    auditoria.to_csv(ruta, index=False, encoding="utf-8-sig")

    print(f"\nAuditoría de fuga guardada en: {ruta}")

    print("\nVariables marcadas como posible fuga:")
    cols_print = [
        "variable",
        "tipo",
        "n_unicos",
        "auc_numerico",
        "auc_modelo_univariado",
        "max_tasa_churn",
        "min_tasa_churn",
        "categoria_max",
        "categoria_min",
        "motivos_fuga",
    ]

    print(
        auditoria[auditoria["posible_fuga"]][cols_print]
        .head(50)
        .to_string(index=False)
    )

    return auditoria

In [ ]:
# ============================================================
# Selección de variables
# ============================================================

def seleccionar_variables_iniciales(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    auditoria: pd.DataFrame,
    modo: str,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series, list[str]]:
    """
    modo:
    - estricto: solo variables demográficas y antigüedad.
    - previas_auditadas: variables candidatas previas que no fueron marcadas como fuga.
    """

    if modo not in {"estricto", "previas_auditadas"}:
        raise ValueError("modo debe ser 'estricto' o 'previas_auditadas'")

    variables_fuga = set(
        auditoria.loc[auditoria["posible_fuga"], "variable"].tolist()
    )

    if modo == "estricto":
        columnas_base = COLUMNAS_ESTRICTAS
    else:
        columnas_base = COLUMNAS_CANDIDATAS_PREVIAS

    columnas_usar = []

    for columna in columnas_base:
        if columna not in train_df.columns:
            continue

        if columna in COLUMNAS_EXCLUIR_SIEMPRE:
            continue

        if columna_excluida_por_patron(columna):
            continue

        if columna in variables_fuga:
            continue

        columnas_usar.append(columna)

    if not columnas_usar:
        raise ValueError(f"No quedaron variables para el modo {modo}")

    X_train = train_df[columnas_usar].copy()
    X_test = test_df[columnas_usar].copy()

    y_train = train_df[COLUMNA_LABEL].copy()
    y_test = test_df[COLUMNA_LABEL].copy()

    print("\n" + "=" * 80)
    print(f"Variables iniciales seleccionadas - modo {modo}")
    print("=" * 80)

    for c in columnas_usar:
        print(f"- {c}")

    print(f"\nTotal variables iniciales: {len(columnas_usar)}")

    pd.DataFrame({"variable": columnas_usar}).to_csv(
        CARPETA_REPORTES / f"variables_iniciales_{modo}.csv",
        index=False,
        encoding="utf-8-sig",
    )

    return X_train, X_test, y_train, y_test, columnas_usar



In [ ]:
# ============================================================
# Preprocesamiento y modelos
# ============================================================

def separar_tipos_columnas(
    X: pd.DataFrame,
) -> tuple[list[str], list[str]]:
    columnas_numericas = X.select_dtypes(include=["number", "bool"]).columns.tolist()
    columnas_categoricas = X.select_dtypes(include=["object", "category"]).columns.tolist()

    return columnas_numericas, columnas_categoricas


def crear_preprocesador(
    columnas_numericas: list[str],
    columnas_categoricas: list[str],
    escalar_numericas: bool,
) -> ColumnTransformer:
    transformers = []

    if columnas_numericas:
        pasos_num = [("imputer", SimpleImputer(strategy="median"))]

        if escalar_numericas:
            pasos_num.append(("scaler", StandardScaler()))

        transformers.append(
            (
                "num",
                Pipeline(steps=pasos_num),
                columnas_numericas,
            )
        )

    if columnas_categoricas:
        transformers.append(
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", crear_one_hot_encoder()),
                    ]
                ),
                columnas_categoricas,
            )
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )


def crear_modelos(
    columnas_numericas: list[str],
    columnas_categoricas: list[str],
) -> dict[str, Pipeline]:
    modelos: dict[str, Pipeline] = {}

    modelos["dummy_baseline"] = Pipeline(
        steps=[
            (
                "preprocesador",
                crear_preprocesador(
                    columnas_numericas,
                    columnas_categoricas,
                    escalar_numericas=False,
                ),
            ),
            ("modelo", DummyClassifier(strategy="most_frequent")),
        ]
    )

    modelos["regresion_logistica"] = Pipeline(
        steps=[
            (
                "preprocesador",
                crear_preprocesador(
                    columnas_numericas,
                    columnas_categoricas,
                    escalar_numericas=True,
                ),
            ),
            (
                "modelo",
                LogisticRegression(
                    max_iter=1000,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    )

    modelos["random_forest"] = Pipeline(
        steps=[
            (
                "preprocesador",
                crear_preprocesador(
                    columnas_numericas,
                    columnas_categoricas,
                    escalar_numericas=False,
                ),
            ),
            (
                "modelo",
                RandomForestClassifier(
                    n_estimators=250,
                    max_depth=6,
                    min_samples_leaf=150,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    )

    modelos["extra_trees"] = Pipeline(
        steps=[
            (
                "preprocesador",
                crear_preprocesador(
                    columnas_numericas,
                    columnas_categoricas,
                    escalar_numericas=False,
                ),
            ),
            (
                "modelo",
                ExtraTreesClassifier(
                    n_estimators=250,
                    max_depth=6,
                    min_samples_leaf=150,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    )

    modelos["hist_gradient_boosting"] = Pipeline(
        steps=[
            (
                "preprocesador",
                crear_preprocesador(
                    columnas_numericas,
                    columnas_categoricas,
                    escalar_numericas=False,
                ),
            ),
            (
                "modelo",
                HistGradientBoostingClassifier(
                    max_iter=200,
                    max_leaf_nodes=15,
                    learning_rate=0.05,
                    l2_regularization=1.0,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

    return modelos

In [ ]:
# ============================================================
# Nombres e importancia de variables
# ============================================================

def obtener_nombres_features(
    modelo: Pipeline,
    columnas_numericas: list[str],
    columnas_categoricas: list[str],
) -> list[str]:
    preprocesador = modelo.named_steps["preprocesador"]

    try:
        return preprocesador.get_feature_names_out().tolist()
    except Exception:
        nombres = []

        for c in columnas_numericas:
            nombres.append(f"num__{c}")

        if columnas_categoricas and "cat" in preprocesador.named_transformers_:
            encoder = preprocesador.named_transformers_["cat"].named_steps["onehot"]

            try:
                cat_names = encoder.get_feature_names_out(columnas_categoricas).tolist()
                nombres.extend([f"cat__{n}" for n in cat_names])
            except Exception:
                nombres.extend([f"cat__{c}" for c in columnas_categoricas])

        return nombres


def mapear_feature_a_variable_base(
    feature_name: str,
    columnas_numericas: list[str],
    columnas_categoricas: list[str],
) -> str:
    if feature_name.startswith("num__"):
        return feature_name.replace("num__", "", 1)

    if feature_name.startswith("cat__"):
        limpio = feature_name.replace("cat__", "", 1)

        for col in sorted(columnas_categoricas, key=len, reverse=True):
            if limpio == col or limpio.startswith(f"{col}_"):
                return col

        return limpio

    for col in columnas_numericas + columnas_categoricas:
        if feature_name == col or feature_name.startswith(f"{col}_"):
            return col

    return feature_name


def importancia_agregada_por_variable(
    modelo: Pipeline,
    columnas_numericas: list[str],
    columnas_categoricas: list[str],
) -> pd.DataFrame:
    estimador = modelo.named_steps["modelo"]

    if isinstance(estimador, DummyClassifier):
        return pd.DataFrame(columns=["variable", "importancia"])

    nombres = obtener_nombres_features(
        modelo=modelo,
        columnas_numericas=columnas_numericas,
        columnas_categoricas=columnas_categoricas,
    )

    if hasattr(estimador, "feature_importances_"):
        valores = estimador.feature_importances_
    elif hasattr(estimador, "coef_"):
        valores = np.abs(estimador.coef_[0])
    else:
        return pd.DataFrame(columns=["variable", "importancia"])

    longitud = min(len(nombres), len(valores))

    filas = []

    for nombre, valor in zip(nombres[:longitud], valores[:longitud]):
        variable_base = mapear_feature_a_variable_base(
            feature_name=nombre,
            columnas_numericas=columnas_numericas,
            columnas_categoricas=columnas_categoricas,
        )

        filas.append(
            {
                "feature_transformada": nombre,
                "variable": variable_base,
                "importancia": float(valor),
            }
        )

    df_imp = pd.DataFrame(filas)

    if df_imp.empty:
        return pd.DataFrame(columns=["variable", "importancia"])

    df_base = (
        df_imp.groupby("variable", as_index=False)["importancia"]
        .sum()
        .sort_values("importancia", ascending=False)
    )

    return df_base



In [ ]:
# ============================================================
# Poda iterativa de fuga
# ============================================================

def entrenar_modelo_detector_fuga(
    X_train: pd.DataFrame,
    y_train: pd.Series,
) -> tuple[Pipeline, float, list[str], list[str]]:
    columnas_numericas, columnas_categoricas = separar_tipos_columnas(X_train)

    X_a, X_b, y_a, y_b = train_test_split(
        X_train,
        y_train,
        test_size=VALIDATION_SIZE_FOR_PRUNING,
        random_state=RANDOM_STATE,
        stratify=y_train,
    )

    X_a_bal, y_a_bal = balancear_train(
        X_train=X_a,
        y_train=y_a,
        metodo=BALANCE_METHOD,
    )

    modelo = Pipeline(
        steps=[
            (
                "preprocesador",
                crear_preprocesador(
                    columnas_numericas,
                    columnas_categoricas,
                    escalar_numericas=False,
                ),
            ),
            (
                "modelo",
                RandomForestClassifier(
                    n_estimators=200,
                    max_depth=5,
                    min_samples_leaf=150,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    )

    modelo.fit(X_a_bal, y_a_bal)

    proba = modelo.predict_proba(X_b)[:, 1]
    auc = roc_auc_score(y_b, proba)

    return modelo, float(auc), columnas_numericas, columnas_categoricas


def podar_variables_por_fuga_iterativa(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    columnas_iniciales: list[str],
    modo: str,
) -> list[str]:
    if not ACTIVAR_PODA_ITERATIVA:
        return columnas_iniciales

    columnas_actuales = columnas_iniciales.copy()
    registros = []

    print("\n" + "=" * 80)
    print(f"Poda iterativa de fuga - modo {modo}")
    print("=" * 80)

    for iteracion in range(1, MAX_ITERACIONES_PODA + 1):
        if len(columnas_actuales) <= MIN_VARIABLES_DESPUES_PODA:
            print("\nSe detiene la poda porque se llegó al mínimo de variables permitido.")
            break

        X_actual = X_train[columnas_actuales].copy()

        modelo_detector, auc_val, cols_num, cols_cat = entrenar_modelo_detector_fuga(
            X_train=X_actual,
            y_train=y_train,
        )

        df_imp = importancia_agregada_por_variable(
            modelo=modelo_detector,
            columnas_numericas=cols_num,
            columnas_categoricas=cols_cat,
        )

        if df_imp.empty:
            print("No se pudo calcular importancia para la poda.")
            break

        variable_top = str(df_imp.iloc[0]["variable"])
        importancia_top = float(df_imp.iloc[0]["importancia"])

        registros.append(
            {
                "iteracion": iteracion,
                "auc_validacion": auc_val,
                "variable_removida": variable_top if auc_val >= AUC_OBJETIVO_PODA else "",
                "importancia_variable_removida": importancia_top,
                "n_variables_antes": len(columnas_actuales),
            }
        )

        print(
            f"Iteración {iteracion} | AUC validación: {auc_val:.4f} | "
            f"Top variable: {variable_top} ({importancia_top:.4f})"
        )

        if auc_val < AUC_OBJETIVO_PODA:
            print(
                f"AUC {auc_val:.4f} ya está por debajo del objetivo de poda "
                f"({AUC_OBJETIVO_PODA}). Se detiene."
            )
            break

        if auc_val < AUC_ALERTA_PODA:
            print(
                f"AUC {auc_val:.4f} ya no está en zona de alerta fuerte "
                f"({AUC_ALERTA_PODA}). Se detiene."
            )
            break

        if variable_top not in columnas_actuales:
            print(f"No se puede remover {variable_top} porque no está en columnas_actuales.")
            break

        columnas_actuales.remove(variable_top)

    df_registros = pd.DataFrame(registros)
    ruta_log = CARPETA_REPORTES / f"poda_iterativa_{modo}.csv"
    df_registros.to_csv(ruta_log, index=False, encoding="utf-8-sig")

    ruta_vars = CARPETA_REPORTES / f"variables_finales_despues_poda_{modo}.csv"
    pd.DataFrame({"variable": columnas_actuales}).to_csv(
        ruta_vars,
        index=False,
        encoding="utf-8-sig",
    )

    print(f"\nLog de poda guardado en: {ruta_log}")
    print(f"Variables finales guardadas en: {ruta_vars}")

    print("\nVariables finales después de poda:")
    for c in columnas_actuales:
        print(f"- {c}")

    return columnas_actuales

In [ ]:
# ============================================================
# Evaluación
# ============================================================

def obtener_probabilidad_churn(
    modelo: Pipeline,
    X: pd.DataFrame,
) -> np.ndarray:
    estimador = modelo.named_steps["modelo"]

    if hasattr(estimador, "predict_proba"):
        return modelo.predict_proba(X)[:, 1]

    return modelo.predict(X).astype(float)


def evaluar_metricas_clasicas(
    y_real: pd.Series,
    y_pred: np.ndarray,
    y_proba: np.ndarray,
) -> dict[str, float]:
    return {
        "accuracy": float(accuracy_score(y_real, y_pred)),
        "precision": float(precision_score(y_real, y_pred, zero_division=0)),
        "recall": float(recall_score(y_real, y_pred, zero_division=0)),
        "f1": float(f1_score(y_real, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_real, y_proba)),
        "average_precision": float(average_precision_score(y_real, y_proba)),
    }


def evaluar_ranking_churn(
    y_real: pd.Series,
    y_proba: np.ndarray,
) -> pd.DataFrame:
    df_eval = pd.DataFrame(
        {
            "y_real": y_real.values,
            "probabilidad_churn": y_proba,
        }
    ).sort_values("probabilidad_churn", ascending=False)

    total_churn = df_eval["y_real"].sum()
    tasa_base = df_eval["y_real"].mean()

    resultados = []

    for p in TOP_PORCENTAJES:
        n = max(1, int(len(df_eval) * p))
        top = df_eval.head(n)

        churn_top = int(top["y_real"].sum())
        precision_top = float(top["y_real"].mean())
        recall_top = float(churn_top / total_churn) if total_churn > 0 else 0.0
        lift = float(precision_top / tasa_base) if tasa_base > 0 else 0.0

        resultados.append(
            {
                "top_porcentaje": p,
                "clientes_top": n,
                "churn_en_top": churn_top,
                "precision_top": precision_top,
                "recall_top": recall_top,
                "lift": lift,
            }
        )

    return pd.DataFrame(resultados)


def buscar_mejor_umbral_f1(
    y_real: pd.Series,
    y_proba: np.ndarray,
) -> dict[str, float]:
    mejor = {
        "umbral": 0.50,
        "precision": 0.0,
        "recall": 0.0,
        "f1": 0.0,
    }

    for umbral in np.arange(0.05, 0.96, 0.01):
        y_pred = (y_proba >= umbral).astype(int)

        precision = precision_score(y_real, y_pred, zero_division=0)
        recall = recall_score(y_real, y_pred, zero_division=0)
        f1 = f1_score(y_real, y_pred, zero_division=0)

        if f1 > mejor["f1"]:
            mejor = {
                "umbral": float(umbral),
                "precision": float(precision),
                "recall": float(recall),
                "f1": float(f1),
            }

    return mejor


def guardar_matriz_confusion(
    modo: str,
    nombre_modelo: str,
    y_real: pd.Series,
    y_pred: np.ndarray,
    sufijo: str,
) -> None:
    matriz = confusion_matrix(y_real, y_pred)

    df_matriz = pd.DataFrame(
        matriz,
        index=["Real_No_Churn", "Real_Churn"],
        columns=["Pred_No_Churn", "Pred_Churn"],
    )

    ruta_csv = CARPETA_REPORTES / f"matriz_confusion_{modo}_{nombre_modelo}_{sufijo}.csv"
    df_matriz.to_csv(ruta_csv, encoding="utf-8-sig")

    plt.figure(figsize=(6, 5))
    plt.imshow(matriz)
    plt.title(f"Matriz confusión - {modo} - {nombre_modelo} - {sufijo}")
    plt.xlabel("Predicción")
    plt.ylabel("Real")
    plt.xticks([0, 1], ["No churn", "Churn"])
    plt.yticks([0, 1], ["No churn", "Churn"])

    for i in range(matriz.shape[0]):
        for j in range(matriz.shape[1]):
            plt.text(j, i, matriz[i, j], ha="center", va="center")

    plt.tight_layout()

    ruta_png = CARPETA_REPORTES / f"matriz_confusion_{modo}_{nombre_modelo}_{sufijo}.png"
    plt.savefig(ruta_png, dpi=150)
    plt.close()


def guardar_distribucion_probabilidades(
    modo: str,
    nombre_modelo: str,
    y_real: pd.Series,
    y_proba: np.ndarray,
) -> None:
    df_plot = pd.DataFrame(
        {
            "churn_real": y_real.values,
            "probabilidad_churn": y_proba,
        }
    )

    plt.figure(figsize=(8, 5))

    plt.hist(
        df_plot.loc[df_plot["churn_real"] == 0, "probabilidad_churn"],
        bins=30,
        alpha=0.6,
        label="No churn",
    )

    plt.hist(
        df_plot.loc[df_plot["churn_real"] == 1, "probabilidad_churn"],
        bins=30,
        alpha=0.6,
        label="Churn",
    )

    plt.title(f"Distribución probabilidades - {modo} - {nombre_modelo}")
    plt.xlabel("Probabilidad de churn")
    plt.ylabel("Cantidad")
    plt.legend()
    plt.tight_layout()

    ruta = CARPETA_REPORTES / f"distribucion_probabilidades_{modo}_{nombre_modelo}.png"
    plt.savefig(ruta, dpi=150)
    plt.close()


def guardar_predicciones(
    modo: str,
    nombre_modelo: str,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    y_proba: np.ndarray,
    mejor_umbral: float,
    test_df_original: pd.DataFrame,
) -> None:
    y_pred_05 = (y_proba >= 0.50).astype(int)
    y_pred_opt = (y_proba >= mejor_umbral).astype(int)

    resultados = pd.DataFrame(index=X_test.index)

    if "cliente_id" in test_df_original.columns:
        resultados["cliente_id"] = test_df_original.loc[X_test.index, "cliente_id"].values

    resultados["churn_real"] = y_test.values
    resultados["probabilidad_churn"] = y_proba
    resultados["pred_umbral_0_50"] = y_pred_05
    resultados["pred_umbral_optimo"] = y_pred_opt
    resultados["umbral_optimo"] = mejor_umbral

    resultados = resultados.sort_values("probabilidad_churn", ascending=False)

    ruta = CARPETA_SALIDAS / f"predicciones_{modo}_{nombre_modelo}.csv"
    resultados.to_csv(ruta, index=False, encoding="utf-8-sig")

    print(f"Predicciones guardadas: {ruta}")
    print("\nTop 10 clientes con mayor probabilidad:")
    print(resultados.head(10).to_string(index=False))


def guardar_importancias(
    modo: str,
    nombre_modelo: str,
    modelo: Pipeline,
    columnas_numericas: list[str],
    columnas_categoricas: list[str],
) -> None:
    df_imp = importancia_agregada_por_variable(
        modelo=modelo,
        columnas_numericas=columnas_numericas,
        columnas_categoricas=columnas_categoricas,
    )

    if df_imp.empty:
        return

    ruta_csv = CARPETA_REPORTES / f"importancia_variables_{modo}_{nombre_modelo}.csv"
    df_imp.to_csv(ruta_csv, index=False, encoding="utf-8-sig")

    top_20 = df_imp.head(20).sort_values("importancia", ascending=True)

    plt.figure(figsize=(10, 8))
    plt.barh(top_20["variable"], top_20["importancia"])
    plt.title(f"Top variables - {modo} - {nombre_modelo}")
    plt.xlabel("Importancia agregada")
    plt.tight_layout()

    ruta_png = CARPETA_REPORTES / f"importancia_variables_{modo}_{nombre_modelo}.png"
    plt.savefig(ruta_png, dpi=150)
    plt.close()


In [ ]:
# ============================================================
# Sanidad con target aleatorio
# ============================================================

def prueba_target_aleatorio(
    modo: str,
    X_train: pd.DataFrame,
    y_train: pd.Series,
) -> dict[str, Any]:
    print("\n" + "#" * 80)
    print(f"Prueba de sanidad target aleatorio - {modo}")
    print("#" * 80)

    columnas_numericas, columnas_categoricas = separar_tipos_columnas(X_train)

    y_random = y_train.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    X_random = X_train.reset_index(drop=True)

    X_a, X_b, y_a, y_b = train_test_split(
        X_random,
        y_random,
        test_size=0.30,
        random_state=RANDOM_STATE,
        stratify=y_random,
    )

    X_a_bal, y_a_bal = balancear_train(
        X_train=X_a,
        y_train=y_a,
        metodo=BALANCE_METHOD,
    )

    modelo = Pipeline(
        steps=[
            (
                "preprocesador",
                crear_preprocesador(
                    columnas_numericas,
                    columnas_categoricas,
                    escalar_numericas=True,
                ),
            ),
            (
                "modelo",
                LogisticRegression(
                    max_iter=1000,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    )

    modelo.fit(X_a_bal, y_a_bal)

    proba = modelo.predict_proba(X_b)[:, 1]
    pred = (proba >= 0.50).astype(int)

    resultado = {
        "modo": modo,
        "accuracy_random": float(accuracy_score(y_b, pred)),
        "f1_random": float(f1_score(y_b, pred, zero_division=0)),
        "roc_auc_random": float(roc_auc_score(y_b, proba)),
    }

    print(resultado)

    if resultado["roc_auc_random"] > 0.60:
        print("ALERTA: el target aleatorio dio AUC alto. Revisar pipeline.")
    else:
        print("OK: target aleatorio cerca de 0.50. El pipeline parece sano.")

    return resultado


In [ ]:
# ============================================================
# Entrenamiento y evaluación
# ============================================================

def entrenar_y_evaluar_modo(
    modo: str,
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    auditoria: pd.DataFrame,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    X_train_ini, X_test_ini, y_train, y_test, columnas_ini = seleccionar_variables_iniciales(
        train_df=train_df,
        test_df=test_df,
        auditoria=auditoria,
        modo=modo,
    )

    columnas_finales = podar_variables_por_fuga_iterativa(
        X_train=X_train_ini,
        y_train=y_train,
        columnas_iniciales=columnas_ini,
        modo=modo,
    )

    X_train = X_train_ini[columnas_finales].copy()
    X_test = X_test_ini[columnas_finales].copy()

    columnas_numericas, columnas_categoricas = separar_tipos_columnas(X_train)

    resultado_random = prueba_target_aleatorio(
        modo=modo,
        X_train=X_train,
        y_train=y_train,
    )

    X_train_bal, y_train_bal = balancear_train(
        X_train=X_train,
        y_train=y_train,
        metodo=BALANCE_METHOD,
    )

    modelos = crear_modelos(
        columnas_numericas=columnas_numericas,
        columnas_categoricas=columnas_categoricas,
    )

    metricas_modo = []

    for nombre_modelo, modelo in modelos.items():
        print("\n" + "#" * 80)
        print(f"Entrenando modelo: {modo} - {nombre_modelo}")
        print("#" * 80)

        modelo.fit(X_train_bal, y_train_bal)

        y_proba = obtener_probabilidad_churn(modelo, X_test)
        y_pred_05 = (y_proba >= 0.50).astype(int)

        metricas = evaluar_metricas_clasicas(
            y_real=y_test,
            y_pred=y_pred_05,
            y_proba=y_proba,
        )

        mejor_umbral = buscar_mejor_umbral_f1(
            y_real=y_test,
            y_proba=y_proba,
        )

        y_pred_opt = (y_proba >= mejor_umbral["umbral"]).astype(int)

        metricas_opt = evaluar_metricas_clasicas(
            y_real=y_test,
            y_pred=y_pred_opt,
            y_proba=y_proba,
        )

        ranking = evaluar_ranking_churn(
            y_real=y_test,
            y_proba=y_proba,
        )

        fila = {
            "modo": modo,
            "modelo": nombre_modelo,
            "balance_method": BALANCE_METHOD,
            "n_variables": len(columnas_finales),
            "variables": "|".join(columnas_finales),

            "accuracy_050": metricas["accuracy"],
            "precision_050": metricas["precision"],
            "recall_050": metricas["recall"],
            "f1_050": metricas["f1"],
            "roc_auc": metricas["roc_auc"],
            "average_precision": metricas["average_precision"],

            "mejor_umbral_f1": mejor_umbral["umbral"],
            "precision_umbral_optimo": metricas_opt["precision"],
            "recall_umbral_optimo": metricas_opt["recall"],
            "f1_umbral_optimo": metricas_opt["f1"],
        }

        metricas_modo.append(fila)

        print("\nMétricas con umbral 0.50:")
        for k, v in metricas.items():
            print(f"{k}: {v:.4f}")

        print("\nMejor umbral por F1:")
        print(mejor_umbral)

        print("\nMétricas con umbral óptimo:")
        print(
            {
                "precision": metricas_opt["precision"],
                "recall": metricas_opt["recall"],
                "f1": metricas_opt["f1"],
            }
        )

        print("\nReporte clasificación con umbral 0.50:")
        print(classification_report(y_test, y_pred_05, zero_division=0))

        print("\nMétricas de ranking:")
        print(ranking.to_string(index=False))

        if metricas["roc_auc"] > 0.95:
            print(
                "\nALERTA CRÍTICA: AUC > 0.95. "
                "El modelo sigue probablemente contaminado por fuga de información."
            )

        if metricas["roc_auc"] < 0.55:
            print(
                "\nNota: AUC cercano a 0.50. "
                "El modelo no está encontrando señal predictiva suficiente."
            )

        if 0.60 <= metricas["roc_auc"] <= 0.85:
            print(
                "\nResultado potencialmente defendible: "
                "AUC en rango moderado y menos sospechoso."
            )

        ruta_ranking = CARPETA_REPORTES / f"ranking_{modo}_{nombre_modelo}.csv"
        ranking.to_csv(ruta_ranking, index=False, encoding="utf-8-sig")

        guardar_matriz_confusion(
            modo=modo,
            nombre_modelo=nombre_modelo,
            y_real=y_test,
            y_pred=y_pred_05,
            sufijo="umbral_050",
        )

        guardar_matriz_confusion(
            modo=modo,
            nombre_modelo=nombre_modelo,
            y_real=y_test,
            y_pred=y_pred_opt,
            sufijo="umbral_optimo",
        )

        guardar_distribucion_probabilidades(
            modo=modo,
            nombre_modelo=nombre_modelo,
            y_real=y_test,
            y_proba=y_proba,
        )

        guardar_predicciones(
            modo=modo,
            nombre_modelo=nombre_modelo,
            X_test=X_test,
            y_test=y_test,
            y_proba=y_proba,
            mejor_umbral=mejor_umbral["umbral"],
            test_df_original=test_df,
        )

        guardar_importancias(
            modo=modo,
            nombre_modelo=nombre_modelo,
            modelo=modelo,
            columnas_numericas=columnas_numericas,
            columnas_categoricas=columnas_categoricas,
        )

        ruta_modelo = CARPETA_MODELOS / f"modelo_{modo}_{nombre_modelo}.pkl"
        joblib.dump(modelo, ruta_modelo)

        print(f"Modelo guardado: {ruta_modelo}")

    return metricas_modo, [resultado_random]



In [ ]:
# ============================================================
# Guardado general
# ============================================================

def guardar_resultados_generales(
    metricas: list[dict[str, Any]],
    pruebas_random: list[dict[str, Any]],
) -> None:
    df_metricas = pd.DataFrame(metricas)

    ruta_metricas_csv = CARPETA_REPORTES / "metricas_modelos_v4.csv"
    ruta_metricas_json = CARPETA_REPORTES / "metricas_modelos_v4.json"

    df_metricas.to_csv(ruta_metricas_csv, index=False, encoding="utf-8-sig")

    with open(ruta_metricas_json, "w", encoding="utf-8") as f:
        json.dump(metricas, f, indent=4, ensure_ascii=False)

    df_random = pd.DataFrame(pruebas_random)
    ruta_random = CARPETA_REPORTES / "pruebas_target_aleatorio_v4.csv"
    df_random.to_csv(ruta_random, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 80)
    print("RESUMEN FINAL")
    print("=" * 80)

    columnas_resumen = [
        "modo",
        "modelo",
        "balance_method",
        "n_variables",
        "accuracy_050",
        "precision_050",
        "recall_050",
        "f1_050",
        "roc_auc",
        "average_precision",
        "mejor_umbral_f1",
        "precision_umbral_optimo",
        "recall_umbral_optimo",
        "f1_umbral_optimo",
    ]

    print(df_metricas[columnas_resumen].to_string(index=False))

    print(f"\nMétricas guardadas en: {ruta_metricas_csv}")
    print(f"Pruebas target aleatorio guardadas en: {ruta_random}")


def guardar_documentacion_experimento() -> None:
    texto = f"""
# Modelo churn V4 balanceado y auditado

## Objetivo

Entrenar modelos de churn reduciendo el riesgo de fuga de información y corrigiendo el desbalanceo de clases.

## Decisiones metodológicas

1. El split train/test se hace antes de cualquier balanceo.
2. El balanceo se aplica solamente sobre train.
3. El test conserva la distribución real del problema.
4. La auditoría de fuga se calcula usando únicamente train.
5. Las variables numéricas con baja cardinalidad también se auditan como categóricas.
6. Se aplica poda iterativa si el conjunto de variables sigue produciendo AUC sospechosamente alto.
7. Se comparan varios modelos bajo el mismo conjunto final de variables.

## Balanceo usado

Método configurado:

`{BALANCE_METHOD}`

## Interpretación recomendada

- AUC cercano a 0.50: el modelo no encuentra señal predictiva.
- AUC entre 0.60 y 0.85: resultado más defendible.
- AUC mayor a 0.95: sospecha fuerte de fuga de información.
- Precision@Top y Lift son especialmente importantes para churn, porque permiten priorizar clientes.

## Limitación principal

Este script reduce fugas estadísticas, pero no reemplaza la solución correcta:

Features antes de una fecha de corte -> churn futuro después de esa fecha.

Ejemplo:

Variables hasta 2022-06-30 -> churn entre 2022-07-01 y 2022-09-30.
"""

    ruta = CARPETA_REPORTES / "README_experimento_v4.md"

    with open(ruta, "w", encoding="utf-8") as f:
        f.write(texto.strip())

    print(f"Documentación guardada en: {ruta}")



In [ ]:
# ============================================================
# Main
# ============================================================

def main() -> None:
    crear_carpetas()

    df = cargar_datos()
    df = construir_label(df)

    train_df, test_df = separar_train_test(df)

    auditoria = auditar_variables_train(train_df)

    todas_metricas: list[dict[str, Any]] = []
    todas_pruebas_random: list[dict[str, Any]] = []

    for modo in ["estricto", "previas_auditadas"]:
        metricas_modo, pruebas_random_modo = entrenar_y_evaluar_modo(
            modo=modo,
            train_df=train_df,
            test_df=test_df,
            auditoria=auditoria,
        )

        todas_metricas.extend(metricas_modo)
        todas_pruebas_random.extend(pruebas_random_modo)

    guardar_resultados_generales(
        metricas=todas_metricas,
        pruebas_random=todas_pruebas_random,
    )

    guardar_documentacion_experimento()

    print("\nProceso finalizado correctamente.")


if __name__ == "__main__":
    main()


Dataset cargado
Filas: 248,515
Columnas: 40

Distribución del target:
churn
0    198495
1     50020
Name: count, dtype: int64

Distribución porcentual:
churn
0    79.87
1    20.13
Name: proportion, dtype: float64

Split train/test creado
Train: 198,812
Test: 49,703

Distribución target en train:
churn
0    0.7987
1    0.2013
Name: proportion, dtype: float64

Distribución target en test:
churn
0    0.7987
1    0.2013
Name: proportion, dtype: float64

Auditoría de fuga guardada en: reports_v4_balanceado_auditado\auditoria_fuga_train.csv

Variables marcadas como posible fuga:
                 variable    tipo  n_unicos  auc_numerico  auc_modelo_univariado  max_tasa_churn  min_tasa_churn              categoria_max              categoria_min                                                                                                                                                                      motivos_fuga
     ha_solicitado_retiro   int64         2      1.000000               1.